# 10. Hypothesis tests for the product team

**What:** four questions the business asked, answered with statistics.
**Why:** a model ranks visitors; these tests answer "is group A really better than group B, and by how much?"

For each question: conversion rate per group, the difference with a 95% confidence interval,
a two-proportion z-test, and the same check at **client level** (one row per client),
because sessions of the same client are not independent.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest

df = pd.read_parquet("../data/processed/features_v2.parquet",
                     columns=["client_id", "target", "utm_medium", "device_category",
                              "is_organic", "is_social_ads", "is_presence_city"])
print(f"{len(df):,} sessions | CR {df['target'].mean():.2%}")

def compare(frame, mask_a, mask_b, label_a, label_b):
    a, b = frame.loc[mask_a, "target"], frame.loc[mask_b, "target"]
    pa, pb = a.mean(), b.mean()
    diff = pa - pb
    se = np.sqrt(pa * (1 - pa) / len(a) + pb * (1 - pb) / len(b))
    _, p = proportions_ztest([a.sum(), b.sum()], [len(a), len(b)])
    return {"A": label_a, "B": label_b, "n_A": len(a), "n_B": len(b),
            "CR_A_%": round(pa * 100, 2), "CR_B_%": round(pb * 100, 2),
            "diff_pp": round(diff * 100, 2),
            "CI95_pp": f"[{(diff - 1.96 * se) * 100:.2f}, {(diff + 1.96 * se) * 100:.2f}]",
            "relative_%": round(diff / pb * 100, 1), "p_value": f"{p:.1e}"}

def client_level(frame):
    """One row per client: did the client ever convert? Group = the client's most frequent value."""
    g = frame.groupby("client_id")
    out = g["target"].max().to_frame()
    for c in ["is_organic", "is_social_ads", "is_presence_city"]:
        out[c] = g[c].mean().round()
    out["device_category"] = g["device_category"].agg(lambda s: s.mode().iat[0])
    out["paid"] = 1 - out["is_organic"]
    return out

paid = df["is_organic"] == 0

QUESTIONS = [
    ("Organic vs paid traffic",       lambda f: (f["is_organic"] == 1, f["is_organic"] == 0, "organic", "paid")),
    ("Mobile vs desktop",             lambda f: (f["device_category"] == "mobile", f["device_category"] == "desktop", "mobile", "desktop")),
    ("Moscow & SPb vs other cities",  lambda f: (f["is_presence_city"] == 1, f["is_presence_city"] == 0, "Moscow/SPb", "other")),
    ("Social ads vs other paid",      lambda f: ((f["is_social_ads"] == 1) & (f["is_organic"] == 0),
                                                 (f["is_social_ads"] == 0) & (f["is_organic"] == 0), "social ads", "other paid")),
]

1,732,266 sessions | CR 2.90%


## 1. Session level

In [2]:
session_res = pd.DataFrame([{"question": q, **compare(df, *fn(df))} for q, fn in QUESTIONS])
session_res

,question,A,B,n_A,n_B,CR_A_%,CR_B_%,diff_pp,CI95_pp,relative_%,p_value
0,Organic vs paid traffic,organic,paid,468920,1263346,4.44,2.34,2.10,"[2.04, 2.17]",90.1,0.0e+00
1,Mobile vs desktop,mobile,desktop,1368679,346891,2.80,3.32,-0.51,"[-0.58, -0.45]",-15.5,1.9e-58
2,Moscow & SPb vs other cities,Moscow/SPb,other,1029330,702936,2.99,2.78,0.20,"[0.15, 0.25]",7.3,7.0e-15
3,Social ads vs other paid,social ads,other paid,253547,1009799,1.55,2.53,-0.98,"[-1.03, -0.92]",-38.6,1.9e-186


## 2. Client level (robustness check)

In [3]:
cl = client_level(df)
client_res = pd.DataFrame([{"question": q, **compare(cl, *fn(cl))} for q, fn in QUESTIONS])
client_res[["question", "CR_A_%", "CR_B_%", "diff_pp", "CI95_pp", "relative_%", "p_value"]]

,question,CR_A_%,CR_B_%,diff_pp,CI95_pp,relative_%,p_value
0,Organic vs paid traffic,5.32,2.72,2.60,"[2.51, 2.68]",95.5,0.0e+00
1,Mobile vs desktop,3.22,3.76,-0.54,"[-0.62, -0.46]",-14.4,5.8e-41
2,Moscow & SPb vs other cities,3.47,3.12,0.36,"[0.30, 0.42]",11.4,1.0e-29
3,Social ads vs other paid,1.71,2.99,-1.28,"[-1.34, -1.21]",-42.7,2.1e-231


## 3. Is mobile vs desktop just a traffic-mix effect?
Mobile users may come from different channels. Compare devices **within** each medium.

In [4]:
top_media = df["utm_medium"].value_counts().index[:5]
rows = []
for m in top_media:
    f = df[df["utm_medium"] == m]
    r = compare(f, f["device_category"] == "mobile", f["device_category"] == "desktop", "mobile", "desktop")
    rows.append({"medium": m, "n": len(f), "CR_mobile_%": r["CR_A_%"], "CR_desktop_%": r["CR_B_%"],
                 "diff_pp": r["diff_pp"], "CI95_pp": r["CI95_pp"]})
pd.DataFrame(rows)

,medium,n,CR_mobile_%,CR_desktop_%,diff_pp,CI95_pp
0,banner,525206,2.88,2.22,0.66,"[0.14, 1.17]"
1,cpc,399395,2.36,2.51,-0.15,"[-0.27, -0.04]"
2,(none),277060,3.54,4.55,-1.01,"[-1.18, -0.84]"
3,cpm,229791,1.33,0.45,0.87,"[0.80, 0.95]"
4,referral,136851,5.66,6.36,-0.70,"[-0.96, -0.45]"


In [5]:
session_res.to_csv("../reports/hypotheses_session.csv", index=False)
client_res.to_csv("../reports/hypotheses_client.csv", index=False)